# Issue #14 — Build Standardized Roll Panel

Build the standardized ZN quarterly-roll dataset **using only data already collected by the project**.

This notebook does **not** call Databento or download anything. It starts from the processed panel created in `02_clean_features.ipynb` and the daily OHLCV already saved by `01_collect_data.ipynb`.

The existing pipeline already contains front/next volume, open interest, calendar-spread volume, next-contract volume/OI shares, and `bdays_to_fnd`. This notebook does not rebuild those pieces. It standardizes them, adds price-based liquidity measures available from the existing OHLCV, and saves one final panel under `data/processed/`.

There is no observed bid/ask dataset in the current project files. We therefore do **not** invent bid/ask quotes. For transaction-cost work downstream, we add two explicitly labeled proxies from the data we do have:

- a trailing Roll-style effective-spread estimate from calendar-spread close-price changes;
- half of the observed daily calendar-spread high-low range as a conservative range-based proxy.

Following the liquidity perspective in *Dynamic Hedging*, these should be treated as imperfect execution-cost proxies, not as true slippage for a given order size.

In [7]:
from pathlib import Path
import numpy as np
import pandas as pd

# Locate project root whether the notebook is launched from project root or notebooks/
project_root = next(
    (p for p in (Path.cwd(), Path.cwd().parent)
     if (p / 'data' / 'processed' / 'zn_roll_panel.parquet').exists()),
    None
)
if project_root is None:
    raise FileNotFoundError(
        f"Could not locate data/processed/zn_roll_panel.parquet from {Path.cwd()}"
    )

processed_dir = project_root / 'data' / 'processed'
raw_dir = project_root / 'data' / 'raw'

panel_path = processed_dir / 'zn_roll_panel.parquet'
ohlcv_path = raw_dir / 'zn_ohlcv_daily.parquet'

panel = pd.read_parquet(panel_path)
ohlcv = pd.read_parquet(ohlcv_path)

panel['date'] = pd.to_datetime(panel['date'])
ohlcv['date'] = pd.to_datetime(ohlcv['date'])

required_panel = {
    'roll_id', 'front', 'next', 'date', 'bdays_to_fnd',
    'front_volume', 'next_volume', 'total_volume', 'next_volume_share',
    'front_oi', 'next_oi', 'total_oi', 'next_oi_share', 'spread_volume'
}
missing = required_panel - set(panel.columns)
if missing:
    raise ValueError(f"Existing roll panel is missing columns: {sorted(missing)}")

required_ohlcv = {'date', 'symbol', 'open', 'high', 'low', 'close', 'volume'}
missing = required_ohlcv - set(ohlcv.columns)
if missing:
    raise ValueError(f"Existing OHLCV file is missing columns: {sorted(missing)}")

print(f"Existing panel: {len(panel):,} rows across {panel['roll_id'].nunique()} rolls")
print(f"Existing OHLCV: {len(ohlcv):,} rows")


Existing panel: 837 rows across 27 rolls
Existing OHLCV: 7,916 rows


## 1. Start from the existing processed panel

Issue #14 overlaps heavily with work already completed in `02_clean_features.ipynb`. We preserve those fields rather than recomputing them.

`bdays_to_fnd` is also inherited from the existing panel so this notebook remains consistent with Steps 4–6. Any change to the project's business-day/FND convention should be made upstream, not silently inside this standardization step.

In [8]:
standard = panel.sort_values(['roll_id', 'date']).copy()

# Complementary shares and ratios from fields already present.
standard['front_volume_share'] = 1.0 - standard['next_volume_share']
standard['front_oi_share'] = 1.0 - standard['next_oi_share']

standard['next_to_front_volume_ratio'] = np.where(
    standard['front_volume'] > 0,
    standard['next_volume'] / standard['front_volume'],
    np.nan
)
standard['next_to_front_oi_ratio'] = np.where(
    standard['front_oi'] > 0,
    standard['next_oi'] / standard['front_oi'],
    np.nan
)

# Share of observed roll-related activity represented by the listed calendar spread.
standard['total_roll_activity'] = (
    standard['front_volume'].fillna(0)
    + standard['next_volume'].fillna(0)
    + standard['spread_volume'].fillna(0)
)
standard['spread_volume_share'] = np.where(
    standard['total_roll_activity'] > 0,
    standard['spread_volume'].fillna(0) / standard['total_roll_activity'],
    np.nan
)

display(standard.head())


,roll_id,front,next,date,bdays_to_fnd,front_volume,next_volume,total_volume,next_volume_share,front_oi,next_oi,total_oi,next_oi_share,spread_volume,front_volume_share,front_oi_share,next_to_front_volume_ratio,next_to_front_oi_ratio,total_roll_activity,spread_volume_share
0,ZNH0->ZNM0,ZNH0,ZNM0,2020-01-30,-25,2535625,634,2536259,0.000250,3803706.0,99244.0,3902950.0,0.025428,29640,0.999750,0.974572,0.000250,0.026091,2565899,0.011552
1,ZNH0->ZNM0,ZNH0,ZNM0,2020-01-31,-24,2466911,1076,2467987,0.000436,3832439.0,120691.0,3953130.0,0.030530,27875,0.999564,0.969470,0.000436,0.031492,2495862,0.011168
2,ZNH0->ZNM0,ZNH0,ZNM0,2020-02-02,-23,46012,84,46096,0.001822,3889130.0,142146.0,4031276.0,0.035261,101,0.998178,0.964739,0.001826,0.036550,46197,0.002186
3,ZNH0->ZNM0,ZNH0,ZNM0,2020-02-03,-22,2185444,663,2186107,0.000303,3879566.0,142146.0,4021712.0,0.035345,13305,0.999697,0.964655,0.000303,0.036640,2199412,0.006049
4,ZNH0->ZNM0,ZNH0,ZNM0,2020-02-04,-21,2113255,2765,2116020,0.001307,3909460.0,152276.0,4061736.0,0.037490,10403,0.998693,0.962510,0.001308,0.038951,2126423,0.004892


## 2. Merge price information already present in the raw OHLCV file

The raw OHLCV collected in Step 1 already contains outright contracts and calendar spreads. We only merge those existing observations onto the roll panel.

No network request is made here.

In [9]:
# Make symbol/date unique before merging.
px = (ohlcv[['date', 'symbol', 'open', 'high', 'low', 'close']]
      .sort_values(['symbol', 'date'])
      .drop_duplicates(['symbol', 'date'], keep='last'))

front_px = px.rename(columns={
    'symbol': 'front',
    'open': 'front_open', 'high': 'front_high',
    'low': 'front_low', 'close': 'front_close'
})
next_px = px.rename(columns={
    'symbol': 'next',
    'open': 'next_open', 'high': 'next_high',
    'low': 'next_low', 'close': 'next_close'
})

standard['spread_symbol'] = standard['front'] + '-' + standard['next']
spread_px = px.rename(columns={
    'symbol': 'spread_symbol',
    'open': 'spread_open', 'high': 'spread_high',
    'low': 'spread_low', 'close': 'spread_close'
})

standard = standard.merge(
    front_px, on=['date', 'front'], how='left', validate='many_to_one'
)
standard = standard.merge(
    next_px, on=['date', 'next'], how='left', validate='many_to_one'
)
standard = standard.merge(
    spread_px, on=['date', 'spread_symbol'], how='left', validate='many_to_one'
)

print("Price coverage:")
for c in ['front_close', 'next_close', 'spread_close']:
    print(f"  {c}: {standard[c].notna().mean():.1%}")


Price coverage:
  front_close: 99.9%
  next_close: 100.0%
  spread_close: 99.4%


## 3. Add liquidity measures available from the existing data

Observed bid/ask quotes are unavailable, so we keep the distinction between **observed liquidity variables** and **estimated cost proxies**.

Observed variables:
- front/next volume and OI;
- next-contract volume and OI shares;
- calendar-spread volume and its share of observed roll activity;
- daily high-low ranges.

Estimated proxies:
- `spread_roll_effective_spread_est`: Roll-style implied effective spread from negative first-order covariance of calendar-spread price changes;
- `spread_roll_half_spread_cost_est`: one-half of that estimated spread;
- `spread_half_range_cost_proxy`: one-half of the observed calendar-spread daily high-low range.

The Roll-style estimate is left missing when the estimated serial covariance is nonnegative. A nonnegative covariance does **not** mean transaction cost is zero; it means this estimator does not identify a positive spread for that window.

In [10]:
standard['front_intraday_range'] = (
    standard['front_high'] - standard['front_low']
).clip(lower=0)
standard['next_intraday_range'] = (
    standard['next_high'] - standard['next_low']
).clip(lower=0)
standard['spread_intraday_range'] = (
    standard['spread_high'] - standard['spread_low']
).clip(lower=0)

standard['front_intraday_range_pct'] = np.where(
    standard['front_close'].abs() > 0,
    standard['front_intraday_range'] / standard['front_close'].abs(),
    np.nan
)
standard['next_intraday_range_pct'] = np.where(
    standard['next_close'].abs() > 0,
    standard['next_intraday_range'] / standard['next_close'].abs(),
    np.nan
)

# Conservative, always-transparent range proxy.
standard['spread_half_range_cost_proxy'] = 0.5 * standard['spread_intraday_range']

ROLL_WINDOW = 10
ROLL_MIN_PERIODS = 6

parts = []

for roll_id, grp in standard.groupby('roll_id', sort=False):
    grp = grp.sort_values('date').copy()

    dp = grp['spread_close'].diff()

    # Trailing covariance between consecutive calendar-spread price changes
    cov1 = dp.rolling(
        window=ROLL_WINDOW,
        min_periods=ROLL_MIN_PERIODS
    ).cov(dp.shift(1))

    # Roll-style effective spread estimate
    effective = pd.Series(np.nan, index=grp.index, dtype='float64')

    mask = cov1 < 0
    effective.loc[mask] = 2.0 * np.sqrt(-cov1.loc[mask])

    grp['spread_price_change'] = dp
    grp['spread_return_autocov_1'] = cov1
    grp['spread_roll_effective_spread_est'] = effective
    grp['spread_roll_half_spread_cost_est'] = 0.5 * effective

    # Keep the group identifier explicitly
    grp['roll_id'] = roll_id

    parts.append(grp)

standard = pd.concat(parts, ignore_index=True)

display(standard[[
    'roll_id',
    'date',
    'bdays_to_fnd',
    'next_volume_share',
    'next_oi_share',
    'spread_volume',
    'spread_volume_share',
    'spread_intraday_range',
    'spread_roll_effective_spread_est',
    'spread_roll_half_spread_cost_est',
    'spread_half_range_cost_proxy'
]].head(15))


,roll_id,date,bdays_to_fnd,next_volume_share,next_oi_share,spread_volume,spread_volume_share,spread_intraday_range,spread_roll_effective_spread_est,spread_roll_half_spread_cost_est,spread_half_range_cost_proxy
0,ZNH0->ZNM0,2020-01-30,-25,0.000250,0.025428,29640,0.011552,0.023438,NaN,NaN,0.011719
1,ZNH0->ZNM0,2020-01-31,-24,0.000436,0.030530,27875,0.011168,0.031250,NaN,NaN,0.015625
2,ZNH0->ZNM0,2020-02-02,-23,0.001822,0.035261,101,0.002186,0.015625,NaN,NaN,0.007812
3,ZNH0->ZNM0,2020-02-03,-22,0.000303,0.035345,13305,0.006049,0.023438,NaN,NaN,0.011719
4,ZNH0->ZNM0,2020-02-04,-21,0.001307,0.037490,10403,0.004892,0.023438,NaN,NaN,0.011719
5,ZNH0->ZNM0,2020-02-05,-20,0.001520,0.039337,14775,0.006024,0.031250,NaN,NaN,0.015625
6,ZNH0->ZNM0,2020-02-06,-19,0.000939,0.041924,16474,0.009507,0.023438,NaN,NaN,0.011719
7,ZNH0->ZNM0,2020-02-07,-18,0.001633,0.044894,14847,0.006724,0.015625,NaN,NaN,0.007812
8,ZNH0->ZNM0,2020-02-09,-17,0.005374,0.044865,413,0.005884,0.007812,NaN,NaN,0.003906
9,ZNH0->ZNM0,2020-02-10,-16,0.002976,0.044832,17508,0.011236,0.015625,NaN,NaN,0.007812


## 4. Validate the standardized panel

The point of this notebook is consolidation, not to create a second version of the upstream calculations. The checks below make sure the standardized panel still has one observation per roll/date and that the original share identities are preserved.

In [11]:
duplicates = standard.duplicated(['roll_id', 'date']).sum()
print(f"Duplicate roll/date rows: {duplicates}")

vol_share_error = (
    standard['front_volume_share'] + standard['next_volume_share'] - 1.0
).abs()
oi_share_error = (
    standard['front_oi_share'] + standard['next_oi_share'] - 1.0
).abs()

print(f"Max volume-share identity error: {vol_share_error.dropna().max():.3e}")
print(f"Max OI-share identity error: {oi_share_error.dropna().max():.3e}")
print(
    "Roll-style cost estimate coverage: "
    f"{standard['spread_roll_half_spread_cost_est'].notna().mean():.1%}"
)
print(
    "Range-cost proxy coverage: "
    f"{standard['spread_half_range_cost_proxy'].notna().mean():.1%}"
)

summary_cols = [
    'next_volume_share', 'next_oi_share', 'spread_volume',
    'spread_volume_share', 'front_intraday_range_pct',
    'next_intraday_range_pct', 'spread_intraday_range',
    'spread_roll_half_spread_cost_est', 'spread_half_range_cost_proxy'
]
display(standard[summary_cols].describe().T)


Duplicate roll/date rows: 0
Max volume-share identity error: 0.000e+00
Max OI-share identity error: 0.000e+00
Roll-style cost estimate coverage: 52.6%
Range-cost proxy coverage: 99.4%


,count,mean,std,min,25%,50%,75%,max
next_volume_share,837.0,0.280882,0.400728,0.000024,0.002260,0.018302,0.592915,1.000000e+00
next_oi_share,813.0,0.313882,0.409100,0.002079,0.010960,0.039572,0.801061,9.988392e-01
spread_volume,837.0,275853.649940,659057.903444,0.000000,3149.000000,11453.000000,94498.000000,4.645750e+06
spread_volume_share,837.0,0.094126,0.166957,0.000000,0.003009,0.010562,0.087834,6.969427e-01
front_intraday_range_pct,836.0,0.004477,0.002874,0.000000,0.002489,0.004046,0.006027,1.959422e-02
next_intraday_range_pct,837.0,0.004472,0.002900,0.000000,0.002475,0.003999,0.006021,1.912417e-02
spread_intraday_range,832.0,0.028480,0.020877,0.000000,0.015625,0.023438,0.039062,2.031250e-01
spread_roll_half_spread_cost_est,440.0,0.007547,0.004645,0.000824,0.004118,0.006484,0.009407,2.409382e-02
spread_half_range_cost_proxy,832.0,0.014240,0.010439,0.000000,0.007812,0.011719,0.019531,1.015625e-01


## Interpretation

The original roll panel already contained the main migration variables: front- and next-contract volume, open interest, next-contract shares, calendar-spread volume, and timing relative to FND. The purpose of this step was therefore not to rebuild the earlier work, but to put those variables into one standardized panel and add liquidity measures that can be constructed from the data already collected.

The front, next, and calendar-spread price series have nearly complete coverage: 99.9%, 100.0%, and 99.4%, respectively. This allows us to supplement volume and open-interest measures with price-based measures of execution conditions.

The Roll-style half-spread estimate has only 52.6% coverage because it is defined only when the estimated first-order covariance of spread-price changes is negative. Its median value is approximately 0.0065 price points. The high-low based proxy has 99.4% coverage and a median of approximately 0.0117 price points.

These measures should not be interpreted as observed bid-ask spreads or realized slippage. We do not have order-book data, so they are proxies for the cost and difficulty of execution. This distinction matters because high trading volume alone does not guarantee that a given position can be executed cheaply.

## 5. Save the processed standardized panel

This is the Issue #14 output. The file stays under `data/processed/`; it does not need to be committed if the project's data directory is intentionally ignored by Git.

In [12]:
output_path = processed_dir / 'zn_standardized_roll_panel.parquet'
standard.to_parquet(output_path, index=False)

print(f"Saved {len(standard):,} rows across {standard['roll_id'].nunique()} rolls")



Saved 837 rows across 27 rolls
